In [1]:
import common as cm
import pandas as pd
import os
import numpy as np

# Process all available trait in AoU

In [2]:
# read results from different methods
def read_results(method):
    auc = pd.read_csv(os.path.join(cm.ROOT_PATH, "result", "ensembleprs", f"aou_{method}_result.csv"))
    return auc

autogluon_auc = read_results("autogluon")
tabpfn_auc = read_results("tabpfn")
glm_auc = read_results("glm")
superlearner_auc = read_results("superlearner")
single_auc = read_results("bestsingle")
targetsingle_auc = read_results("besttargetsingle")

# All-PRS GLM results (one row per ICD; "auc" is the all-PRS ensemble AUC)
allprs_auc = pd.read_csv(os.path.join(cm.ROOT_PATH, "result", "increase_prs_num", "aou_glm_allprs_result.csv"))

In [3]:
common_models = set(autogluon_auc["icd"]) & set(tabpfn_auc["icd"]) & set(glm_auc["icd"]) & set(superlearner_auc["icd"]) & set(targetsingle_auc["icd"]) & set(single_auc["icd"]) 
autogluon_auc = autogluon_auc[autogluon_auc["icd"].isin(common_models)]
tabpfn_auc = tabpfn_auc[tabpfn_auc["icd"].isin(common_models)]
glm_auc = glm_auc[glm_auc["icd"].isin(common_models)]
superlearner_auc = superlearner_auc[superlearner_auc["icd"].isin(common_models)]
targetsingle_auc = targetsingle_auc[targetsingle_auc["icd"].isin(common_models)]
single_auc = single_auc[single_auc["icd"].isin(common_models)]

print("Number of available traits (ICD10CM):", len(common_models))
print("Number of available traits (ICD10CM):", len(autogluon_auc), len(tabpfn_auc), len(glm_auc), len(superlearner_auc), len(targetsingle_auc), len(single_auc))

# Check how many common traits also have all-PRS GLM results
print("Common traits with all-PRS results:", len(set(allprs_auc["icd"]) & common_models))

Number of available traits (ICD10CM): 367
Number of available traits (ICD10CM): 367 367 367 367 367 367
Common traits with all-PRS results: 367


In [6]:
### generate summary file for all traits to summary_reports ###
case_info = pd.read_csv(os.path.join(cm.ROOT_PATH, "reference_data", "ICD10CM_summary.csv"))

# Directory and file name pattern of per-trait PRS ranking files (edit to actual location)
ranking_dir = os.path.join(cm.ROOT_PATH, "prs_ranking", "AoU_training")

rows = []
for idx, icd in enumerate(common_models):
    description = case_info[case_info["icd_root"] == icd]["description"].iloc[0]

    # Read ensemble training case number from the trait's ranking file (NaN if file missing)
    ranking_file = os.path.join(ranking_dir, f"{icd}.csv")
    train_case_num = pd.read_csv(ranking_file, usecols=["n_cases"])["n_cases"].iloc[0]

    rows.append({
        "icd": icd,
        "gwas_case_num": case_info[case_info["icd_root"] == icd]["n_case"].iloc[0],
        "ensemble_case_num": train_case_num,
        "autogluon": autogluon_auc[autogluon_auc["icd"] == icd]["auc"].iloc[0],
        "tabpfn": tabpfn_auc[tabpfn_auc["icd"] == icd]["auc"].iloc[0],
        "glm": glm_auc[glm_auc["icd"] == icd]["auc"].iloc[0],
        "superlearner": superlearner_auc[superlearner_auc["icd"] == icd]["auc"].iloc[0],
        "best_target_single": targetsingle_auc[targetsingle_auc["icd"] == icd]["auc"].iloc[0],
        "best_all_single": single_auc[single_auc["icd"] == icd]["auc"].iloc[0],
        "all_prs": allprs_auc[allprs_auc["icd"] == icd]["auc"].iloc[0],
        "description": str(description)
    })

report = pd.DataFrame(rows)[["icd", "gwas_case_num", "ensemble_case_num", "autogluon", "tabpfn", "glm",
                             "superlearner", "best_target_single", "best_all_single", "all_prs", "description"]]
report_file = os.path.join(cm.ROOT_PATH, "result", "ensembleprs", "aou_ensembleprs_overall_alltraits.csv")
report.to_csv(report_file, index=False)

# Process overlap traits in AoU

In [7]:
# read common model with UKBB validation set
common_models = pd.read_csv(os.path.join(cm.ROOT_PATH, "result", "ensembleprs", f"ukb_glm_result.csv"))["icd"]

# get intersection of models
autogluon_auc = autogluon_auc[autogluon_auc["icd"].isin(common_models)]
tabpfn_auc = tabpfn_auc[tabpfn_auc["icd"].isin(common_models)]
glm_auc = glm_auc[glm_auc["icd"].isin(common_models)]
superlearner_auc = superlearner_auc[superlearner_auc["icd"].isin(common_models)]
targetsingle_auc = targetsingle_auc[targetsingle_auc["icd"].isin(common_models)]
single_auc = single_auc[single_auc["icd"].isin(common_models)]

print("Number of available traits (ICD10CM):", len(common_models))
print("Number of available traits (ICD10CM):", len(autogluon_auc), len(tabpfn_auc), len(glm_auc), len(superlearner_auc), len(targetsingle_auc), len(single_auc))

Number of available traits (ICD10CM): 325
Number of available traits (ICD10CM): 325 325 325 325 325 325


In [9]:
### generate summary file to summary_reports ###
case_info = pd.read_csv(os.path.join(cm.ROOT_PATH, "reference_data", "ICD10CM_summary.csv"))

# Directory and file name pattern of per-trait PRS ranking files (edit to actual location)
ranking_dir = os.path.join(cm.ROOT_PATH, "prs_ranking", "AoU_training")

rows = []
for idx, icd in enumerate(common_models):
    description = case_info[case_info["icd_root"] == icd]["description"].iloc[0]

    # Read ensemble training case number from the trait's ranking file (NaN if file missing)
    ranking_file = os.path.join(ranking_dir, f"{icd}.csv")
    train_case_num = pd.read_csv(ranking_file, usecols=["n_cases"])["n_cases"].iloc[0]
      
    rows.append({
        "icd": icd,
        "gwas_case_num": case_info[case_info["icd_root"] == icd]["n_case"].iloc[0],
        "ensemble_case_num": train_case_num,
        "autogluon": autogluon_auc[autogluon_auc["icd"] == icd]["auc"].iloc[0],
        "tabpfn": tabpfn_auc[tabpfn_auc["icd"] == icd]["auc"].iloc[0],
        "glm": glm_auc[glm_auc["icd"] == icd]["auc"].iloc[0],
        "superlearner": superlearner_auc[superlearner_auc["icd"] == icd]["auc"].iloc[0],
        "best_target_single": targetsingle_auc[targetsingle_auc["icd"] == icd]["auc"].iloc[0],
        "best_all_single": single_auc[single_auc["icd"] == icd]["auc"].iloc[0],
        "all_prs": allprs_auc[allprs_auc["icd"] == icd]["auc"].iloc[0],
        "description": description
    })

report = pd.DataFrame(rows)[["icd", "gwas_case_num", "ensemble_case_num", "autogluon", "tabpfn", "glm",
                             "superlearner", "best_target_single", "best_all_single", "all_prs", "description"]]
report_file = os.path.join(cm.ROOT_PATH, "result", "ensembleprs", "aou_ensembleprs_overall_overlaptraits.csv")
report.to_csv(report_file, index=False)

# Process all available (overlap) traits in UKB

In [10]:
# read results from different methods
def read_results(method):
    auc = pd.read_csv(os.path.join(cm.ROOT_PATH, "result", "ensembleprs", f"ukb_{method}_result.csv"))
    return auc

autogluon_auc = read_results("autogluon")
tabpfn_auc = read_results("tabpfn")
glm_auc = read_results("glm")
superlearner_auc = read_results("superlearner")
single_auc = read_results("bestsingle")
targetsingle_auc = read_results("besttargetsingle")

# All-PRS GLM results (one row per ICD; "auc" is the all-PRS ensemble AUC)
allprs_auc = pd.read_csv(os.path.join(cm.ROOT_PATH, "result", "increase_prs_num", "ukb_glm_allprs_result.csv"))

In [11]:
common_models = set(autogluon_auc["icd"]) & set(tabpfn_auc["icd"]) & set(glm_auc["icd"]) & set(superlearner_auc["icd"]) & set(targetsingle_auc["icd"]) & set(single_auc["icd"]) 
autogluon_auc = autogluon_auc[autogluon_auc["icd"].isin(common_models)]
tabpfn_auc = tabpfn_auc[tabpfn_auc["icd"].isin(common_models)]
glm_auc = glm_auc[glm_auc["icd"].isin(common_models)]
superlearner_auc = superlearner_auc[superlearner_auc["icd"].isin(common_models)]
targetsingle_auc = targetsingle_auc[targetsingle_auc["icd"].isin(common_models)]
single_auc = single_auc[single_auc["icd"].isin(common_models)]

print("Number of available traits (ICD10CM):", len(common_models))
print("Number of available traits (ICD10CM):", len(autogluon_auc), len(tabpfn_auc), len(glm_auc), len(superlearner_auc), len(targetsingle_auc), len(single_auc))

# Check how many common traits also have all-PRS GLM results
print("Common traits with all-PRS results:", len(set(allprs_auc["icd"]) & common_models))

Number of available traits (ICD10CM): 325
Number of available traits (ICD10CM): 325 325 325 325 325 325
Common traits with all-PRS results: 325


In [12]:
### generate summary file for all traits to summary_reports ###
case_info = pd.read_csv(os.path.join(cm.ROOT_PATH, "reference_data", "ICD10CM_summary.csv"))

# Directory and file name pattern of per-trait PRS ranking files (edit to actual location)
ranking_dir = os.path.join(cm.ROOT_PATH, "prs_ranking", "AoU_training")

rows = []
for idx, icd in enumerate(common_models):
    description = case_info[case_info["icd_root"] == icd]["description"].iloc[0]

    # Read ensemble training case number from the trait's ranking file (NaN if file missing)
    ranking_file = os.path.join(ranking_dir, f"{icd}.csv")
    train_case_num = pd.read_csv(ranking_file, usecols=["n_cases"])["n_cases"].iloc[0]
      
    rows.append({
        "icd": icd,
        "gwas_case_num": case_info[case_info["icd_root"] == icd]["n_case"].iloc[0],
        "ensemble_case_num": train_case_num,
        "autogluon": autogluon_auc[autogluon_auc["icd"] == icd]["auc"].iloc[0],
        "tabpfn": tabpfn_auc[tabpfn_auc["icd"] == icd]["auc"].iloc[0],
        "glm": glm_auc[glm_auc["icd"] == icd]["auc"].iloc[0],
        "superlearner": superlearner_auc[superlearner_auc["icd"] == icd]["auc"].iloc[0],
        "best_target_single": targetsingle_auc[targetsingle_auc["icd"] == icd]["auc"].iloc[0],
        "best_all_single": single_auc[single_auc["icd"] == icd]["auc"].iloc[0],
        "all_prs": allprs_auc[allprs_auc["icd"] == icd]["auc"].iloc[0],
        "description": description
    })

report = pd.DataFrame(rows)[["icd", "gwas_case_num", "ensemble_case_num", "autogluon", "tabpfn", "glm",
                             "superlearner", "best_target_single", "best_all_single", "all_prs", "description"]]
report_file = os.path.join(cm.ROOT_PATH, "result", "ensembleprs", "ukb_ensembleprs_overall_overlaptraits.csv")
report.to_csv(report_file, index=False)